# Tuần 7 — Chuẩn hóa, kiểm tra chất lượng và đóng gói Dataset v1

Notebook dùng bộ `reference_aligned` tuần 6 làm chuẩn đánh giá mô hình. Thống kê chuẩn hóa chỉ được fit trên Train, sau đó giữ nguyên cho Validation/Test.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mitdb').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục mitdb ở thư mục hiện tại hoặc thư mục cha.')
WEEK3_DIR = PROJECT_DIR / 'Tuần 3-4' / 'ket_qua_ecg'
WEEK6_DIR = PROJECT_DIR / 'Tuần 5-6-7' / 'ket_qua_tuan_6'
RESULT_DIR = PROJECT_DIR / 'Tuần 5-6-7' / 'ket_qua_tuan_7'
DATASET_DIR = PROJECT_DIR / 'Tuần 5-6-7' / 'dataset_v1'
FIG_DIR = RESULT_DIR / 'figures'
RESULT_DIR.mkdir(parents=True, exist_ok=True); DATASET_DIR.mkdir(exist_ok=True); FIG_DIR.mkdir(exist_ok=True)
SOURCE_FILE = WEEK6_DIR / 'aami_reference_aligned.npz'
source = np.load(SOURCE_FILE)
CLASSES = source['classes'].tolist()
data = {split: {'x': source[f'x_{split}'].astype(np.float32), 'y': source[f'y_{split}'], 'record': source[f'record_{split}']}
        for split in ('train', 'val', 'test')}

# Chuẩn hóa global z-score: mean/std được fit duy nhất từ train để tránh rò rỉ dữ liệu.
train_mean = float(data['train']['x'].mean())
train_std = float(data['train']['x'].std())
if not np.isfinite(train_std) or train_std <= 0:
    raise ValueError('Độ lệch chuẩn Train không hợp lệ.')
for split in data:
    data[split]['x_normalized'] = ((data[split]['x'] - train_mean) / train_std).astype(np.float32)

# Kiểm tra cấu trúc, split và dữ liệu số.
split_spec = json.loads((WEEK6_DIR / 'split_validation_checks.json').read_text(encoding='utf-8'))
allowed = set(CLASSES)
checks = []
for split, item in data.items():
    x, y, records = item['x_normalized'], item['y'], item['record']
    checks.append({'check': f'{split}: đúng 180 mẫu/nhịp', 'passed': bool(x.ndim == 2 and x.shape[1] == 180), 'detail': str(x.shape)})
    checks.append({'check': f'{split}: không có NaN/Inf', 'passed': bool(np.isfinite(x).all()), 'detail': f'finite={np.isfinite(x).sum()}/{x.size}'})
    checks.append({'check': f'{split}: nhãn hợp lệ', 'passed': bool(set(y).issubset(allowed)), 'detail': str(sorted(set(y)))})
    checks.append({'check': f'{split}: record đúng split', 'passed': bool(set(records).issubset(set(split_spec[f'{split}_records']))), 'detail': str(sorted(set(records)))})
patient_values = {split: set(split_spec[f'{split}_patients']) for split in ('train', 'val', 'test')}
checks.append({'check': 'Không giao nhau bệnh nhân giữa ba split', 'passed': bool(not (patient_values['train'] & patient_values['val'] or patient_values['train'] & patient_values['test'] or patient_values['val'] & patient_values['test'])), 'detail': 'train∩val=train∩test=val∩test=∅'})
checks_df = pd.DataFrame(checks)
assert checks_df['passed'].all(), checks_df[~checks_df['passed']].to_string(index=False)
print('TẤT CẢ KIỂM TRA CHẤT LƯỢNG ĐỀU ĐẠT')
print(checks_df.to_string(index=False))

# Bảng phân bố lớp và thống kê chuẩn hóa.
class_rows = []
normalization_rows = []
for split, item in data.items():
    counts = Counter(item['y'].tolist())
    for label in CLASSES:
        class_rows.append({'split': split, 'AAMI_class': label, 'beats': counts[label]})
    normalization_rows.append({'split': split, 'mean_before': float(item['x'].mean()), 'std_before': float(item['x'].std()),
                               'mean_after': float(item['x_normalized'].mean()), 'std_after': float(item['x_normalized'].std())})
class_distribution = pd.DataFrame(class_rows)
normalization_stats = pd.DataFrame(normalization_rows)
print('Thống kê trước/sau chuẩn hóa:'); print(normalization_stats.round(5).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for split, color in zip(('train', 'val', 'test'), ('tab:blue', 'tab:orange', 'tab:green')):
    sample = data[split]['x'].ravel()[::100]
    axes[0].hist(sample, bins=80, density=True, alpha=.45, label=split, color=color)
    sample_norm = data[split]['x_normalized'].ravel()[::100]
    axes[1].hist(sample_norm, bins=80, density=True, alpha=.45, label=split, color=color)
axes[0].set_title('Biên độ ECG trước chuẩn hóa'); axes[1].set_title('Biên độ ECG sau chuẩn hóa theo Train')
for ax in axes: ax.set_xlabel('Biên độ'); ax.set_ylabel('Mật độ'); ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(FIG_DIR / 'phan_bo_truoc_sau_chuan_hoa.png', dpi=180); plt.show()

# Vẽ một waveform ngẫu nhiên có seed cố định của từng lớp để kiểm tra trực quan.
rng = np.random.default_rng(20261105)
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(12, 10), sharex=True)
for ax, label in zip(axes, CLASSES):
    candidates = np.flatnonzero(data['train']['y'] == label)
    index = int(rng.choice(candidates))
    ax.plot(np.arange(180), data['train']['x_normalized'][index], color='tab:blue', lw=1)
    ax.axvline(90, color='crimson', ls='--', lw=1, label='Vị trí R')
    ax.set_title(f'Lớp {label} — record {data["train"]["record"][index]}')
    ax.set_ylabel('z-score'); ax.grid(alpha=.25); ax.legend(loc='upper right')
axes[-1].set_xlabel('Chỉ số mẫu trong nhịp 180 mẫu')
fig.tight_layout(); fig.savefig(FIG_DIR / 'waveform_mau_theo_lop.png', dpi=180); plt.show()

# Đóng gói từng split độc lập để giai đoạn CNN chỉ cần đọc train/val/test.
for split, item in data.items():
    np.savez_compressed(DATASET_DIR / f'{split}.npz', x=item['x_normalized'], y=item['y'], record=item['record'], classes=np.asarray(CLASSES, dtype='U1'))
normalization = {'method': 'global_zscore_fit_on_train_only', 'train_mean': train_mean, 'train_std': train_std,
                 'formula': 'x_normalized = (x - train_mean) / train_std'}
manifest_week3 = json.loads((WEEK3_DIR / 'manifest.json').read_text(encoding='utf-8'))
manifest = {'dataset_name': 'ECG_AAMI_Dataset_v1_reference_aligned', 'source_dataset': str(SOURCE_FILE.relative_to(PROJECT_DIR)),
            'classes': CLASSES, 'beat_samples': 180, 'r_peak_index': 90,
            'filter': {'type': 'Butterworth zero-phase bandpass', 'hz': manifest_week3['filter_hz'], 'order': manifest_week3['filter_order']},
            'split_file': str((WEEK6_DIR / 'split_train_val_test.csv').relative_to(PROJECT_DIR)),
            'normalization_file': 'normalization.json', 'split_sizes': {split: int(len(item['y'])) for split, item in data.items()},
            'notes': ['Reference-aligned data is for classifier evaluation.', 'Detector-aligned data remains in ket_qua_tuan_6 for later end-to-end evaluation.', 'Paced records 102, 104, 107, 217 are excluded.']}
(DATASET_DIR / 'normalization.json').write_text(json.dumps(normalization, ensure_ascii=False, indent=2), encoding='utf-8')
(DATASET_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
class_distribution.to_csv(DATASET_DIR / 'class_distribution.csv', index=False, encoding='utf-8-sig')
checks_df.to_csv(RESULT_DIR / 'quality_check_table.csv', index=False, encoding='utf-8-sig')
normalization_stats.to_csv(RESULT_DIR / 'normalization_statistics.csv', index=False, encoding='utf-8-sig')
(RESULT_DIR / 'quality_check_report.md').write_text('# Kiểm tra chất lượng Dataset v1\n\nTất cả kiểm tra cấu trúc, giá trị số, nhãn và split đều đạt. Chuẩn hóa global z-score được fit trên Train duy nhất; Validation/Test không tham gia tính mean/std.\n', encoding='utf-8')
(DATASET_DIR / 'README.md').write_text('# Dataset v1\n\n- `train.npz`, `val.npz`, `test.npz`: mảng `x` (nhịp 180 mẫu đã chuẩn hóa), `y` (AAMI N/S/V/F/Q), `record`.\n- Mở `manifest.json` và `normalization.json` để xem cấu hình.\n- Tái tạo: chạy notebook tuần 3–4, sau đó notebook tuần 6 và notebook tuần 7 theo thứ tự.\n', encoding='utf-8')
print(f'Đã đóng gói Dataset v1 tại: {DATASET_DIR}')
